In [1]:
# Student Name: Aaron Baggot
# Student Number: C22716399
# TU856/4

In [9]:
'''

Underwater Shark Enhancement and Isolation

Tehe assingment provides a single pass that accepts an image as input and 
produces one enhanced image of the shark on a clean, 
minimal bavkground as output. The design generalises to similar images shark underwater or cat in snow.

Algorithm's used
Edge preserving denoise (bi-lateral filter)
The 



'''

"\n\nUnderwater Shark Enhancement and Isolation\n\nTehe assingment provides a single pass that accepts an image as input and \nproduces one enhanced image of the shark on a clean, \nminimal bavkground as output. The design generalises to similar images shark underwater or cat in snow.\n\nAlgorithm's used\nEdge preserving denoise (bi-lateral filter)\nThe \n\n\n\n"

In [10]:
from pathlib import Path
import os

# If we're inside the notebooks/ folder, move up to the project root
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(str(cwd.parent))
print("CWD now:", os.getcwd())  # should end with Image_Processing_Assignment_1

CWD now: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1


In [11]:
from pathlib import Path
import os

print("CWD:", os.getcwd())
raw = Path("data/raw")
print("data/raw exists? ", raw.exists())
print("Files I can see in data/raw:")
for p in raw.glob("*"):
    print(" -", p.name)

# Try to detect the shark file automatically (case/space/extension tolerant)
candidates = []
patterns = [
    "Shark 2.*", "Shark_2.*", "shark 2.*", "shark_2.*",
    "Shark2.*", "shark2.*", "Shark 2*"
]
ex = [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"]

for pat in patterns:
    for p in raw.glob(pat):
        if p.suffix in ex:
            candidates.append(p)

if candidates:
    INPUT_PATH = sorted(candidates, key=lambda p: p.name.lower())[0]
    print("Auto-selected:", INPUT_PATH)
else:
    # Fallback: keep whatever you already had, but warn
    INPUT_PATH = Path("data/raw/Shark 2.PNG")
    print("No candidates found. Expecting:", INPUT_PATH)

CWD: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1
data/raw exists?  True
Files I can see in data/raw:
 - snow_cat.jpg
 - Shark 2.PNG
 - Shark 1.PNG
Auto-selected: data/raw/Shark 2.PNG


In [12]:
from pathlib import Path
import cv2 as cv
import numpy as np

INPUT_PATH  = Path("data/raw/Shark 2.PNG")     
OUTPUT_PATH = Path("outputs/fish_final.png")    # final image file path

def process_image(input_path: Path, output_path: Path):
    I = cv.imread(str(input_path), cv.IMREAD_COLOR)
    if I is None:
        raise FileNotFoundError(f"Input image not found: {input_path}")
    h, w = I.shape[:2]

    # 1) Edge-preserving denoise (bilateral)
    den = cv.bilateralFilter(I, d=7, sigmaColor=75, sigmaSpace=75)

    # 2) Local contrast (CLAHE on LAB L)
    LAB = cv.cvtColor(den, cv.COLOR_BGR2LAB)
    L, A, B = cv.split(LAB)
    L2 = cv.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(L)
    enh = cv.cvtColor(cv.merge([L2, A, B]), cv.COLOR_LAB2BGR)

    # 3) Light unsharp masking
    blur = cv.GaussianBlur(enh, (0,0), 1.0)
    enh  = cv.addWeighted(enh, 1.4, blur, -0.4, 0)

    # 4) Otsu on HSV-S and LAB-b (+ tiny auto-fallback)
    hsv = cv.cvtColor(enh, cv.COLOR_BGR2HSV); S = hsv[:,:,1]
    lab = cv.cvtColor(enh, cv.COLOR_BGR2LAB); b = lab[:,:,2]
    _, th_s = cv.threshold(S, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    _, th_b = cv.threshold(b, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    mask = cv.bitwise_and(th_s, th_b)

    area_ratio = (mask.sum()/255) / (mask.shape[0]*mask.shape[1])
    if area_ratio < 0.01:  # too small -> loosen
        mask_or = cv.bitwise_or(th_s, th_b)
        area_or = (mask_or.sum()/255) / (mask.shape[0]*mask.shape[1])
        mask = mask_or if area_or > area_ratio else th_b
    elif area_ratio > 0.95:  # too big -> tighten
        _, s_inv = cv.threshold(255 - S, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
        mask = cv.bitwise_and(s_inv, th_b)

    # 5) Morphology + largest component
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (7,7))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=2)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN,  k, iterations=1)

    num, labels = cv.connectedComponents(mask)
    if num > 1:
        sizes = [(labels==i).sum() for i in range(1, num)]
        idx   = 1 + int(np.argmax(sizes))
        mask  = np.where(labels==idx, 255, 0).astype(np.uint8)

    # 6) Auto-rotate with minAreaRect and tight crop
    cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if cnts:
        cnt   = max(cnts, key=cv.contourArea)
        angle = cv.minAreaRect(cnt)[-1]
        if angle < -45: angle += 90
        Mrot  = cv.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        I_rot = cv.warpAffine(enh, Mrot, (w, h), flags=cv.INTER_LINEAR,  borderMode=cv.BORDER_REFLECT)
        m_rot = cv.warpAffine(mask, Mrot, (w, h), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_CONSTANT)
        pts = cv.findNonZero(m_rot)
        x,y,ww,hh = cv.boundingRect(pts) if pts is not None else (0,0,w,h)
        crop = I_rot[y:y+hh, x:x+ww]; mcrp = m_rot[y:y+hh, x:x+ww]
    else:
        crop, mcrp = enh, mask  # fallback (rare)

    # 7) Clean minimal background (solid from border median)
    inv = (mcrp==0).astype(np.uint8)*255
    border = np.zeros_like(inv)
    border[:10,:]  = 255; border[-10:,:] = 255
    border[:,:10]  = 255; border[:,-10:] = 255

    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1,3)
    samples = samples[(samples.sum(1) > 0)]
    color   = tuple(map(int, (np.median(samples,0) if len(samples) else np.array([210,220,230]))))
    canvas  = np.full_like(crop, color, dtype=np.uint8)

    final = cv.add(cv.bitwise_and(crop,  crop,  mask=mcrp),
                   cv.bitwise_and(canvas, canvas, mask=cv.bitwise_not(mcrp)))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv.imwrite(str(output_path), final)
    print(f"[OK] Wrote: {output_path}")
process_image(INPUT_PATH, OUTPUT_PATH)



[OK] Wrote: outputs/fish_final.png
